# unsupervised models

# Clustering Model Training
### K-Means, DBSCAN, and GMM clustering models.


## 1. Import Libraries


In [1]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score
import joblib
import os


## 2. Load Preprocessed Data


In [2]:
df = pd.read_csv('data/processed/final_preprocessed_clustering.csv')
print(f"Dataset shape: {df.shape}")


Dataset shape: (11306, 216)


## 3. Prepare Features


In [3]:
# Exclude ID and date columns

X = df.drop(['Transaction ID', 'Customer ID', 'Transaction Date'], axis=1)
print(f"Features: {X.shape[1]}, Samples: {X.shape[0]}")



Features: 213, Samples: 11306


## 4. Find Optimal K


In [7]:
# Test k values from 2 to 8
scores = []
for k in range(2, 9):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X)
    score = silhouette_score(X,kmeans.labels_)
    scores.append(score)
    print(f"k={k}: Silhouette Score = {score:.3f}")

# Find optimal k
optimal_k = range(2, 9)[np.argmax(scores)]
print(f"\nOptimal k = {optimal_k}")


k=2: Silhouette Score = 0.207
k=3: Silhouette Score = 0.171
k=4: Silhouette Score = 0.152
k=5: Silhouette Score = 0.119
k=6: Silhouette Score = 0.104
k=7: Silhouette Score = 0.103
k=8: Silhouette Score = 0.095

Optimal k = 2


## 5. Train Model


In [8]:
# Train K-Means with optimal k
kmeans_model = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
kmeans_model.fit(X)

# Get cluster labels
cluster_labels = kmeans_model.labels_

print(f"Model trained with k={optimal_k}")
print(f"Silhouette Score: {silhouette_score(X, cluster_labels):.3f}")
print(f"\nCluster distribution:")
print(pd.Series(cluster_labels).value_counts().sort_index())


Model trained with k=2
Silhouette Score: 0.207

Cluster distribution:
0    7049
1    4257
Name: count, dtype: int64


## 6. Save Results


In [9]:
# Create directories
os.makedirs('models', exist_ok=True)

# Save model
joblib.dump(kmeans_model, 'models/kmeans_model.pkl')
print("Model saved: models/kmeans_model.pkl")

# Save cluster labels
pd.DataFrame({'Cluster': cluster_labels}).to_csv('data/processed/kmeans_cluster_labels.csv', index=False)
print("Labels saved: data/processed/kmeans_cluster_labels.csv")

# Save data with clusters
df['Cluster'] = cluster_labels
df.to_csv('data/processed/clustered_data.csv', index=False)
print("Clustered data saved: data/processed/clustered_data.csv")


Model saved: models/kmeans_model.pkl
Labels saved: data/processed/kmeans_cluster_labels.csv
Clustered data saved: data/processed/clustered_data.csv


## DBSCAN Clustering


### 7. Find Optimal Parameters


### Test different eps values (DBSCAN parameter)


In [11]:
eps_values = [0.5, 1.0, 1.5, 2.0, 2.5]
min_samples = 5

best_eps = None
best_score = -1
best_labels = None

print("Testing different eps values...")
for eps in eps_values:
    dbscan = DBSCAN(eps=eps, min_samples=min_samples)
    labels = dbscan.fit_predict(X)
    
    # Count clusters (excluding noise points labeled as -1)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = list(labels).count(-1)
    
    # Calculate silhouette score (only if we have at least 2 clusters and not all noise)
    if n_clusters >= 2 and n_noise < len(labels) * 0.5:
        score = silhouette_score(X, labels)
        print(f"eps={eps}: Clusters={n_clusters}, Noise={n_noise}, Silhouette={score:.3f}")
        
        if score > best_score:
            best_score = score
            best_eps = eps
            best_labels = labels
    else:
        print(f"eps={eps}: Clusters={n_clusters}, Noise={n_noise} (too many noise points)")

if best_eps is not None:
    print(f"\nOptimal eps = {best_eps}")
    optimal_eps = best_eps
else:
    # Default if no good parameters found
    optimal_eps = 1.5
    print(f"\nUsing default eps = {optimal_eps}")



Testing different eps values...
eps=0.5: Clusters=296, Noise=9383 (too many noise points)
eps=1.0: Clusters=354, Noise=2669, Silhouette=-0.062
eps=1.5: Clusters=1, Noise=3 (too many noise points)
eps=2.0: Clusters=1, Noise=0 (too many noise points)
eps=2.5: Clusters=1, Noise=0 (too many noise points)

Optimal eps = 1.0
